# M1 데이터 다루기 — 실습 (W2)

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.

**이 실습이 끝나면**
1. numpy의 shape·axis·브로드캐스팅을 직접 다룬다
2. **세트피스:** 다섯 승객 미니 타이타닉 손계산(평균 30.25·0 대치 24.2 왜곡·groupby 0.667/0.5)을 pandas로 검산한다 ⭐
3. 타이타닉 891명으로 선택·필터·그룹·결측 처리를 하고, EDA 실측(여성 0.742·fare 꼬리)을 그림으로 확인한다

**7단계 멘탈모델 초점:** 데이터 → 표현(특징)

## Part A. numpy 복습
### A-1. 배열·shape·axis

In [ ]:
import numpy as np                            # 수치 배열 계산 도구

a = np.array([1, 2, 3, 4])                    # 1차원 배열
print('배열:', a)
print('shape:', a.shape, '| dtype:', a.dtype) # 모양과 자료형

M = np.arange(12).reshape(3, 4)               # 0~11을 3행 4열로
print(M)
print('첫 행:', M[0])                          # 인덱스 0 = 첫 번째 행
print('두 번째 열:', M[:, 1])                  # 모든 행의 1번 열
print('전체 평균:', M.mean())
print('열별 평균:', M.mean(axis=___))          # ✍️ 빈칸: 열별(특징별) 평균이 나오는 axis — "위에서 아래로 누르기"

### A-2. 브로드캐스팅 & 벡터화

In [ ]:
base = np.array([[10, 20, 30],                # 3x3 행렬
                 [40, 50, 60],
                 [70, 80, 90]])
vec = np.array([1, 2, 3])                     # 길이 3 벡터
print(base + vec)                             # 벡터가 각 행에 자동 복제(브로드캐스팅)
print(base * 2)                               # 스칼라 곱은 전체에 적용(벡터화)

## Part B. 세트피스 — 다섯 승객의 미니 타이타닉 ⭐
reading §5의 표를 코드로 재현합니다. **종이 손계산을 먼저** 하고(평균·중앙값·0 대치·그룹 생존율), 아래로 검산하세요.

In [ ]:
import pandas as pd                           # 표(데이터프레임) 도구

mini = pd.DataFrame({                         # 다섯 승객 미니 타이타닉
    'sex': ['female', 'female', 'female', 'male', 'male'],
    'age': [22.0, 38.0, 26.0, None, 35.0],    # 승객 D의 나이는 빈 칸(NaN)
    'survived': [1, 1, 0, 0, 1]})
print(mini)
print('평균(NaN 자동 제외):', mini['age'].mean())        # 30.25 = 121/4
print('중앙값:', mini['age'].median())                   # 30.5 = (26+35)/2
print('0 대치 후 평균:', mini['age'].fillna(0).mean())   # 24.2 — "0살 승객" 발명 = 왜곡!
filled = mini['age'].fillna(mini['age'].___())           # ✍️ 빈칸: 분포를 존중하는 대치 통계량(0이 아닌 그것)
print('중앙값 대치 후 평균:', filled.mean())             # 30.3 — 원래 30.25와 거의 같음
print(mini.groupby('sex')['survived'].mean())            # female 0.667 / male 0.5 — 손계산 일치

> **검산 포인트:** 평균 **30.25**(NaN 자동 제외)·중앙값 30.5 · **0 대치 → 24.2 왜곡** vs 중앙값 대치 → 30.3 · groupby = "쪼개고 각각 평균"(0.667/0.5) — 전부 손계산과 일치. "함부로 0으로 채우지 말 것"은 **24.2라는 숫자**로 기억하세요.

## Part C. 진짜 타이타닉 891명 불러오기

In [ ]:
import seaborn as sns                         # 예제 데이터셋 제공
df = sns.load_dataset('titanic')              # 타이타닉 승객 데이터
df.head()                                     # 앞 5행 미리보기

### C-2. 크기·정보·결측 현황

In [ ]:
print('행 x 열:', df.shape)                    # (891, 15)
print('age 결측:', df['age'].___().sum())      # ✍️ 빈칸: 빈 칸(NaN) 여부를 True/False로 표시하는 메서드
print('deck 결측:', df['deck'].isnull().sum())  # 688 — 대부분 빈 열(삭제 후보)
df.info()                                     # 열별 자료형·결측 현황 한 번에

## Part D. 선택 · 필터 · 그룹
### D-1. 요약과 필터

In [ ]:
df[['age', 'fare']].describe()                # 나이·요금 요약 통계 (마지막 줄이라 표로 표시)

In [ ]:
high = df[df['age'] > ___]                    # ✍️ 빈칸: 60세 초과 승객 필터(그 나이 숫자)
print('60세 초과 승객 수:', len(high))         # 22명
fem_rate = df.loc[df['sex'] == 'female', 'survived'].mean()  # 여성 승객의 생존율
print('여성 생존율:', round(fem_rate, 3))      # 0.742 (남성은 0.189 — 약 4배)

### D-2. 그룹별 집계 — groupby 실전

In [ ]:
print(df.groupby('class', observed=False)['survived'].mean())  # 등급 계단: 0.630/0.473/0.242
result = df.groupby(___)['survived'].mean()   # ✍️ 빈칸: 성별로 묶을 기준 열 이름(문자열)
print(result)                                 # female 0.742 / male 0.189

## Part E. 결측치 처리 — 중앙값 대치

In [ ]:
print('대치 전 결측:', df['age'].isnull().sum())        # 177
age_filled = df['age'].fillna(df['age'].median())       # 중앙값(28.0)으로 대치 — 세트피스의 교훈
print('대치 후 결측:', age_filled.isnull().sum())       # 0
print('대치 전/후 평균:', round(df['age'].mean(), 2),   # 29.7
      '/', round(age_filled.mean(), 2))                 # 29.36 — 중앙값 대치라 크게 안 흔들림

> **M2 복선:** 지금은 표 **전체**의 중앙값으로 채웠습니다. 다음 주에 train/test를 나누면 "**train의 중앙값으로만**" 채워야 합니다 — 시험지 정보가 새는 문제(누수)를 M2에서 정면으로 다룹니다.

## Part F. 시각화 (EDA)
### F-1. 나이 분포 (히스토그램)

In [ ]:
import matplotlib.pyplot as plt               # 그래프 도구

df['age'].hist(bins=20)                       # 나이를 20개 구간으로
plt.xlabel('age')                             # 축(영어)
plt.ylabel('count')
plt.title('Age distribution')                 # 제목(영어)
plt.show()

### F-2. 등급별 생존율 (막대그래프)

In [ ]:
df.groupby('class', observed=False)['survived'].mean().plot(kind='bar')  # 등급 계단을 막대로
plt.ylabel('survival rate')
plt.title('Survival rate by class')
plt.show()

### F-3. fare의 긴 꼬리 — 평균 ≠ 중앙값

In [ ]:
print('fare 평균:', round(df['fare'].mean(), 2))   # 32.2
print('fare 중앙값:', df['fare'].median())          # 14.4542 — 평균이 두 배 이상!
df[___].hist(bins=30)                          # ✍️ 빈칸: 요금 열 이름(문자열)
plt.xlabel('fare')
plt.title('Fare distribution (long right tail)')
plt.show()                                     # 오른쪽 긴 꼬리 — 소수의 큰 값이 평균을 끌어올림

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "다섯 승객 표에서 0 대치가 평균을 24.2로 만드는 계산을 내가 손으로 해 볼 테니 채점해 줘."
- "`groupby('sex')['survived'].mean()`이 내부에서 뭘 하는지 설명해 볼게 — 허점을 찔러 줘."
- "fare의 평균이 중앙값의 두 배가 넘는 이유를 분포 모양으로 설명해 볼게."
- "이 에러가 무슨 뜻이야? `[에러 붙여넣기]`"

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. numpy의 shape·axis·브로드캐스팅을 다뤘다
2. **세트피스** — 다섯 승객 손계산을 pandas로 검산했다(30.25 / **0 대치 24.2 왜곡** / groupby 0.667·0.5)
3. 타이타닉 891명으로 결측 177 확인·중앙값 대치·EDA 실측(여성 0.742·등급 계단·fare 꼬리)을 했다

**스스로 점검**
- [ ] `M.mean(axis=0)`이 어느 방향 평균인지 안다
- [ ] 0 대치가 왜 왜곡인지 숫자(24.2)로 말할 수 있다
- [ ] groupby의 내부 동작을 손계산으로 재현할 수 있다
- [ ] fare의 평균≠중앙값이 무엇을 말하는지 안다
- [ ] "train의 중앙값으로만"이 왜 필요한지 다음 주에 배울 준비가 됐다

**🔹심화 (선택)**
- 나이를 10살 단위 구간(`pd.cut`)으로 나눠 구간별 생존율을 구해 보세요.
- `df.groupby(['sex', 'class'], observed=False)['survived'].mean()`으로 2중 그룹을 만들어 보세요 — 어느 조합이 가장 생존율이 높나요?
- age 히스토그램을 대치 전/후로 겹쳐 그려, 중앙값 대치가 분포에 만드는 "봉우리"를 관찰해 보세요.

**다음 시간(M2):** 이 표로 공정하게 배우고 평가하기 — train/test 분할·과적합·누수.